In [19]:
# %%writefile ../data/download_data.py
import os
import zipfile
import kagglehub  # pip install kagglehub

def download_data(dataset_slug: str = "iamsouravbanerjee/animal-image-dataset-90-different-animals",
                   dest_dir: str = "../data/animals"):
    
    'we use this function for downloading our dara from kagglehub'
    os.makedirs(dest_dir, exist_ok=True)

    # starting the downlaoding
    cache_path = kagglehub.dataset_download(dataset_slug)
    print(f"Downloaded to the path : {cache_path}")

    # extracting the file
    for f in os.listdir(cache_path):
        if f.endswith(".zip"):
            with zipfile.ZipFile(os.path.join(cache_path, f), "r") as z:
                z.extractall(dest_dir)
    else:
        if not os.listdir(dest_dir):
            os.symlink(cache_path, dest_dir + "_link")

    return dest_dir


Writing ../data/download_data.py


In [ ]:
downlaod_data()

In [20]:
# %%writefile ../src/model.py

import torch
import torchvision
from torch import nn
def create_effnetB2_model(num_of_classes = 90 , 
                          seed = 42):
  

  # First we download our model EfficientNet_B2_Weights
  weights = torchvision.models.EfficientNet_B2_Weights.DEFAULT
  transform = weights.transform
  model = torchvision.models.efficientnet_b2(weights=weights)

  # 4. Freeze all layers in base model
  for param in model.parameters():
    param.requires_grad = False

  torch.manual_seed(42)
  torch.cuda.manual_seed(42)
  model.classifier = torch.nn.Sequential(
      nn.Dropout(p=0.3, inplace=True),
      nn.Linear(in_features=1408, out_features=num_of_classes),
  )


Writing ../src/model.py


In [ ]:
# !pip install torchinfo

from torchinfo import summary
from src.model import create_effnetB2_model
effnet_b2_model , effnet_b2_transform= create_effnetB2_model()

# Print EffNetB2 model summary (uncomment for full output) 
summary(effnet_b2_model, 
        input_size=(1, 3, 224, 224),
        col_names=["input_size", "output_size", "num_params", "trainable"],
        col_width=20,
        row_settings=["var_names"])

In [21]:
# %%writefile ../src/data_setup.py
import os
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

num_of_workers = os.cpu_count()

# we use this function to make our test and train Dataloaders

def create_dataloader(transform : int , 
                      batch_size: int , 
                      train_dir:str ,
                      num_of_workers = num_of_workers):
  
  train_data =  datasets.ImageFolder(train_dir,transform=transform)

  classes_name = train_data.classes 

  train_Dataloder = DataLoader(
      train_data ,
      batch_size = batch_size , 
      shuffle = True , 
      num_of_workers = num_of_workers
  )

  return train_Dataloder , classes_name
  
  

Writing ../src/data_setup.py


In [22]:
# %%writefile ../src/train.py
import torch
from torch import nn

from src.engine import train
from src.data_setup import create_dataloader
from src.model import create_effnetB2_model

#import the model and transform function 
effnet_b2_model , effnet_b2_transform= create_effnetB2_model()

train_dir = 'data/animals_link/animals/animals'

device = 'cuda' if torch.cuda.is_available() else 'cpu' #set the device

#set the optimizer
optimizer = torch.optim.Adam(params=effnet_b2_model.parameters(),
                             lr=1e-3)
# Setup loss function
loss_fn = torch.nn.CrossEntropyLoss()

#first we get our train Dataloader and classes
train_Dataloader , classes_name = create_dataloader(transform=effnet_b2_transform , train_dir=train_dir , batch_size=32)

effnetb2_results = train(model=effnet_b2_model,
                                train_dataloader=train_Dataloader,
                                epochs=10,
                                optimizer=optimizer,
                                loss_fn=loss_fn,
                                device=device)

Writing ../src/train.py


In [23]:
%%writefile ../src/utils.py

import torch
import os 
import matplotlib.pyplot as plt


def save_model(model):
    if not os.path.exists('../models'):
        os.makedirs('../models')
        print('models folder created')
        
    torch.save(model.state_dict() , '../models/model.pth')
    print("✅ Model Saved")

def plotresults(train_accs , train_losses):
    plt.figure(figsize=(12,5))

    plt.subplot(1,2,1)
    plt.plot(train_losses, label='Train Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Training Loss')
    plt.legend()

    plt.subplot(1,2,2)
    plt.plot(train_accs, label='Train Accuracy', color='orange')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy (%)')
    plt.title('Training Accuracy')
    plt.legend()

    plt.tight_layout()
    plt.show()

Writing ../src/utils.py


In [ ]:
save_model(effnet_b2_model)

In [ ]:
plotresults(train_accs=effnetb2_results["train_acc"] , train_losses=effnetb2_results["train_loss"])

In [14]:
# now we want to use the model on our data to see the result
from pathlib import Path
from ../src/model import create_effnetB2_model


import torch

from PIL import Image
from timeit import default_timer as timer 
from tqdm.auto import tqdm
from typing import List, Dict


model = create_effnetB2_model()
test_data_path = list(Path('data/animals_link/animals/animals').glob("*/*.jpg"))
test_data_path[:5]



# 1. Create a function to return a list of dictionaries with sample, truth label, prediction, prediction probability and prediction time
def pred_and_store():
    pass

ImportError: attempted relative import with no known parent package